# Run CheckAMG annotate and de-novo on soil and human gut viral datasets

In [20]:
! pip install polars --quiet
! conda install -c bioconda seqkit -y --quiet

done
Solving environment: ...working... done

# All requested packages already installed.



## Data source

[MetaVR (IMG/VR v5)](https://www.meta-virome.org/) ([Fiamenghi et al. (2026) *Nucleic Acids Res.*](https://doi.org/10.1093/nar/gkaf1283)) will be the source database for virus genomes.

* Obtained a list of soil UViGs (environmental; terrestrial; soil) from the [MetaVR website](https://www.meta-virome.org/Uvigs?hieco=Environmental&hicat=Terrestrial&hitype=Soil&pageSize=20) and human gut UViGs (host-associated; mammals (human); digestive system) from the [MetaVR website](https://www.meta-virome.org/Uvigs?hieco=Host-associated&hicat=Mammals%3A%20Human&hitype=Digestive%20system&pageSize=20), then exported the tables with identifying information and metadata:
    * Soil: 2,054,095 viral genome sequences (1,345,242 vOTUs) in `./data/metavr_uvigs_soil.csv.gz`
    * Human gut: 499,416 viral genome sequences (239,755 vOTUs) in `./data/metavr_uvigs_gut.csv.gz`
* The nucleotide sequences of the MetaVR viruses (`IMGVR5_UViG.fna`) can be downlaoded from [meta-virome.org/Downloads](https://www.meta-virome.org/Downloads) but is too large to be included in this repository
* A key that matches the FASTA headers to the sequence names in the metadata will also be needed to match the metadata properly. This doesn't come with the MetaVR downloads but it can be constructed by parsing the `uvig` column in the metadata tables and string matching to headers in `IMGVR5_UViG.fna`. This also is not included here because of the file size.

Quality filtering will be applied below.

## Load metadata and prepare datasets

In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(10)

polars.config.Config

In [2]:
from pathlib import Path
METAVR_DIR = Path("/storage2/databases/metaVR")
METAVR_UVIG_FNA = METAVR_DIR / "IMGVR5_UViG.fna"
MAIN_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript")
DATA_DIR = Path("./data")

In [3]:
metavr_uvig_to_header = (
    pl.read_csv(
        METAVR_DIR / "uvig_to_fastaheader.tsv",
        separator="\t",
        has_header=False,
        new_columns=["uvig", "fasta_header"],
    )
)
metavr_uvig_to_header

uvig,fasta_header
str,str
"""IMGVR_UViG_2582581227_000001""","""IMGVR_UViG_2582581227_000001|2582581227|2582690522"""
"""IMGVR_UViG_2582581228_000001""","""IMGVR_UViG_2582581228_000001|2582581228|2582690523"""
"""IMGVR_UViG_2582581229_000001""","""IMGVR_UViG_2582581229_000001|2582581229|2582690524"""
"""IMGVR_UViG_2582581230_000001""","""IMGVR_UViG_2582581230_000001|2582581230|2582690525"""
"""IMGVR_UViG_2582581231_000001""","""IMGVR_UViG_2582581231_000001|2582581231|2582690526"""
…,…
"""IMGVR_UViG_GVMAG-M-3300047504-3""","""IMGVR_UViG_GVMAG-M-3300047504-3|3300047504|Ga0494773_000172"""
"""IMGVR_UViG_GVMAG-M-3300047504-3""","""IMGVR_UViG_GVMAG-M-3300047504-3|3300047504|Ga0494773_000174"""
"""IMGVR_UViG_GVMAG-M-3300047504-3""","""IMGVR_UViG_GVMAG-M-3300047504-3|3300047504|Ga0494773_000181"""


In [4]:
metavr_uvigs_soil = (
    pl.read_csv(
        DATA_DIR / "metavr_uvigs_soil.csv.gz",
        null_values=["NA", "na", "NaN", "nan", ""],
        infer_schema_length=10000000,
    )
    .join(metavr_uvig_to_header, on="uvig", how="left")
)

In [5]:
metavr_uvigs_soil

uvig,taxon_oid,discovery,discovery_doi,uvig_topology,is_concatemer,quality,checkv_contamination,completeness_method,estimated_completeness,genomad_score,uvig_cds_gene_count,uvig_trna_gene_count,uvig_total_gene_count,uvig_length,votu,n_hallmarks,viral_confidence,ictv_taxonomy,ictv_taxonomy_method,genome_type,source,genetic_code,host_taxonomy,host_taxonomy_method,fasta_header
str,i64,str,str,str,bool,str,f64,str,f64,f64,i64,i64,i64,i64,str,i64,str,str,str,str,str,str,str,str,str
"""IMGVR_UViG_2088090008_000001""",2088090008,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (medium-confidence)""",41.99,0.9555,22,0,22,15761,"""vOTU_00686457""",5,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""",null,null,"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932"""
"""IMGVR_UViG_2088090008_000002""",2088090008,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""HMM-based (lower-bound)""",13.25,0.9799,13,0,13,12365,"""vOTU_00686460""",1,"""Low""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Chloroflexota;c__Limnocylindria;o__Limnocylindrales;f__CSP1-4;g__SPCO01""","""iPHoP""","""IMGVR_UViG_2088090008_000002|2088090008|P3_DRAFT_NODE_6148_len_12315_cov_71_695900"""
"""IMGVR_UViG_2088090008_000003""",2088090008,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""HMM-based (lower-bound)""",9.47,0.9537,21,0,21,9806,"""vOTU_00686459""",0,"""Low""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""",null,null,"""IMGVR_UViG_2088090008_000003|2088090008|P3_DRAFT_NODE_2952_len_9756_cov_85_330666"""
"""IMGVR_UViG_2088090008_000004""",2088090008,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Not-determined""",0.0,null,null,0.8037,10,0,10,5539,"""vOTU_00686458""",0,"""Low""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""",null,null,"""IMGVR_UViG_2088090008_000004|2088090008|P3_DRAFT_NODE_223157_len_5489_cov_17_049189"""
"""IMGVR_UViG_2088090014_000002""",2088090014,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (medium-confidence)""",17.51,0.9586,16,0,16,12617,"""vOTU_00686492""",1,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Actinomycetota;c__Actinomycetes;o__Actinomycetales;f__Micrococcaceae""","""iPHoP""","""IMGVR_UViG_2088090014_000002|2088090014|GPIPI_16496951"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_GVMAG-S-3300050407-46""",3300050407,"""GVMAGs (2025)""",null,"""Virus MAG""",false,"""Low-quality""",0.0,"""GVClass""",33.33,0.6053,228,0,228,213782,"""vOTU_12830369""",1,"""Low""","""r__Varidnaviria;k__Bamfordvirae;p__Nucleocytoviricota;c__Megaviricetes;o__Pimascovirales;f__;g__;s__""","""GVClass""","""dsDNA""","""Metagenome""","""106""",null,null,"""IMGVR_UViG_GVMAG-S-3300050407-46|3300050407|Ga0530106_1564"""
"""IMGVR_UViG_GVMAG-S-3300050407-46""",3300050407,"""GVMAGs (2025)""",null,"""Virus MAG""",false,"""Low-quality""",0.0,"""GVClass""",33.33,0.6053,228,0,228,213782,"""vOTU_12830369""",1,"""Low""","""r__Varidnaviria;k__Bamfordvirae;p__Nucleocytoviricota;c__Megaviricetes;o__Pimascovirales;f__;g__;s__""","""GVClass""","""dsDNA""","""Metagenome""","""106""",null,null,"""IMGVR_UViG_GVMAG-S-3300050407-46|3300050407|Ga0530106_2127"""
"""IMGVR_UViG_GVMAG-S-3300050407-46""",3300050407,"""GVMAGs (2025)"

In [6]:
metavr_uvigs_gut = (
    pl.read_csv(
        DATA_DIR / "metavr_uvigs_gut.csv.gz",
        null_values=["NA", "na", "NaN", "nan", ""],
        infer_schema_length=10000000,
    )
    .join(metavr_uvig_to_header, on="uvig", how="left")
)

In [7]:
metavr_uvigs_gut

uvig,taxon_oid,discovery,discovery_doi,uvig_topology,is_concatemer,quality,checkv_contamination,completeness_method,estimated_completeness,genomad_score,uvig_cds_gene_count,uvig_trna_gene_count,uvig_total_gene_count,uvig_length,votu,n_hallmarks,viral_confidence,ictv_taxonomy,ictv_taxonomy_method,genome_type,source,genetic_code,host_taxonomy,host_taxonomy_method,fasta_header
str,i64,str,str,str,bool,str,f64,str,f64,f64,i64,i64,i64,i64,str,i64,str,str,str,str,str,str,str,str,str
"""IMGVR_UViG_2042536001_000001""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",9.88,0.9828,11,0,11,5931,"""vOTU_00679922""",3,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacillota;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae""","""iPHoP""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274"""
"""IMGVR_UViG_2042536001_000002""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",5.25,0.9834,6,0,6,5113,"""vOTU_00263288""",1,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__Crassvirales;f__Intestivirida…","""vOTU""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Chitinophagales;f__Chitinophagaceae;g__Chitinophaga;s_…","""vOTU""","""IMGVR_UViG_2042536001_000002|2042536001|Irish_EM03_contig03465"""
"""IMGVR_UViG_2042536001_000003""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",12.31,0.9824,10,0,10,5808,"""vOTU_00679921""",0,"""Low""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacillota;c__Clostridia;o__Oscillospirales;f__Oscillospiraceae;g__Vescimonas""","""iPHoP""","""IMGVR_UViG_2042536001_000003|2042536001|Irish_EM03_contig08621"""
"""IMGVR_UViG_2042536001_000004""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",9.66,0.9826,13,0,13,5815,"""vOTU_00679916""",3,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides""","""iPHoP""","""IMGVR_UViG_2042536001_000004|2042536001|Irish_EM03_contig24613"""
"""IMGVR_UViG_2042536001_000005""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",5.39,0.9819,4,0,4,5726,"""vOTU_00679907""",0,"""Low""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__Crassvirales;f__Intestivirida…","""geNomad""","""dsDNA""","""Metagenome""","""11""",null,null,"""IMGVR_UViG_2042536001_000005|2042536001|Irish_EM03_contig06838"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_GVMAG-M-3300039323-104""",3300039323,"""GVMAGs (2025)""",null,"""Virus MAG""",false,"""Not-determined""",0.0,"""GVClass""",0.0,0.0156,253,3,256,276811,"""vOTU_12830872""",1,"""Low""",null,null,null,"""Metagenome""","""4""",null,null,"""IMGVR_UViG_GVMAG-M-3300039323-104|3300039323|Ga0169852_00358"""
"""IMGVR_UViG_GVMAG-M-3300039323-104""",3300039323,"""GVMAGs (2025)""",null,"""Virus MAG""",false,"""Not-determined""",0.0,"""GVClass""",0.0,0.0156,253,3,256,276811,"""vOTU_12830872""",1,"""Low""",null,null,null,"""Metagenome""","""4""",null,null,"""IMGVR_UViG_GVMAG-M-3300039323-104|3300039323|Ga0169852_00197"""
"""IMGVR_UViG_GVMAG-M-3300039323-104""",3300039323,"""GVMAGs (2025)""",null,"""Virus MAG""",false,"""Not-determined""

## Quality-filter the UViGs

Following the guidelines used by [Graham et al. (2024) *Nat. Microbiol.*](https://doi.org/10.1038/s41564-024-01686-x):

>Virus genomes were selected using the following rules:
>
>(1) contigs of at least 1 kb with high similarity to genomes in the CheckV database (that is, that had high- or medium-quality completeness estimates) or that contained direct terminal repeats were automatically selected;
>
>(2) contigs longer than 10 kb were required to have a geNomad virus score higher than 0.8 and to either encode one virus hallmark (for example, terminase, capsid proteins, portal protein and so on), as determined by geNomad, or to have a geNomad virus marker of at least 5.0;
>
>(3) contigs shorter than 10 kb and longer than 5 kb were required to have a geNomad virus score higher than 0.9, to encode at least one virus hallmark and to have a virus marker enrichment higher than 2.0.

Except **(1)** the metadata for MetaVR does not include columns for "geNomad virus marker" or "virus marker enrichment" so those will be ignored, and **(2)** the above guidelines are for non-proviruses, which will be included here, so proviruses will also be kept if they are at least 10 kb, are CheckV complete, medium-m or high-quality, have a CheckV completeness < 5%, have a geNomad score > 0.8, and have at least 1 viral hallmark gene.

In [8]:
metavr_quality_filter = (
    # GSVA filtering rules (no proviruses)
    (
        (pl.col("uvig_topology").is_in(["Linear", "Direct terminal repeat", "Inverted terminal repeat"])) &
        (       
            (
                (pl.col("uvig_length") >= 1_000) &
                (
                    (pl.col("quality").is_in(["Complete", "High-quality", "Medium-quality"])) |
                    (pl.col("uvig_topology").is_in(["Direct terminal repeat", "Inverted terminal repeat"]))
                )
            ) |
            (
                (pl.col("uvig_length") >= 10_000) &
                (pl.col("genomad_score") > 0.8) &
                (pl.col("n_hallmarks") >= 1) # Metadata does not have "genomad_virus_marker" column
            ) |
            (
                (pl.col("uvig_length") < 10_000) & (pl.col("uvig_length") > 5_000) &
                (pl.col("genomad_score") > 0.9) &
                (pl.col("n_hallmarks") >= 1) # Metadata does not have "genomad_marker_enrichment" column
            )
        )
    ) |
    # Provirus filtering
    (
        (pl.col("uvig_topology").is_in(["Provirus"])) &
        (
            (pl.col("uvig_length") >= 10_000) &
            (pl.col("quality").is_in(["Complete", "High-quality", "Medium-quality"])) &
            (pl.col("checkv_contamination") < 0.05) &
            (pl.col("genomad_score") > 0.8) &
            (pl.col("n_hallmarks") >= 1)
        )
    )
)

### MetaVR soil

In [9]:
metavr_uvigs_soil_filtered = metavr_uvigs_soil.filter(metavr_quality_filter)

In [10]:
metavr_uvigs_soil_filtered

uvig,taxon_oid,discovery,discovery_doi,uvig_topology,is_concatemer,quality,checkv_contamination,completeness_method,estimated_completeness,genomad_score,uvig_cds_gene_count,uvig_trna_gene_count,uvig_total_gene_count,uvig_length,votu,n_hallmarks,viral_confidence,ictv_taxonomy,ictv_taxonomy_method,genome_type,source,genetic_code,host_taxonomy,host_taxonomy_method,fasta_header
str,i64,str,str,str,bool,str,f64,str,f64,f64,i64,i64,i64,i64,str,i64,str,str,str,str,str,str,str,str,str
"""IMGVR_UViG_2088090008_000001""",2088090008,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (medium-confidence)""",41.99,0.9555,22,0,22,15761,"""vOTU_00686457""",5,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""",null,null,"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932"""
"""IMGVR_UViG_2088090008_000002""",2088090008,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""HMM-based (lower-bound)""",13.25,0.9799,13,0,13,12365,"""vOTU_00686460""",1,"""Low""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Chloroflexota;c__Limnocylindria;o__Limnocylindrales;f__CSP1-4;g__SPCO01""","""iPHoP""","""IMGVR_UViG_2088090008_000002|2088090008|P3_DRAFT_NODE_6148_len_12315_cov_71_695900"""
"""IMGVR_UViG_2088090014_000002""",2088090014,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (medium-confidence)""",17.51,0.9586,16,0,16,12617,"""vOTU_00686492""",1,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Actinomycetota;c__Actinomycetes;o__Actinomycetales;f__Micrococcaceae""","""iPHoP""","""IMGVR_UViG_2088090014_000002|2088090014|GPIPI_16496951"""
"""IMGVR_UViG_2088090014_000008""",2088090014,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",13.38,0.9807,12,0,12,8032,"""vOTU_00686524""",5,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""",null,null,"""IMGVR_UViG_2088090014_000008|2088090014|GPIPI_16804765"""
"""IMGVR_UViG_2088090014_000009""",2088090014,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",25.32,0.9291,24,0,24,9998,"""vOTU_00686505""",2,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Pseudomonadota;c__Alphaproteobacteria;o__Rhodospirillales;f__UXAT02""","""iPHoP""","""IMGVR_UViG_2088090014_000009|2088090014|GPIPI_16855418"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_8123596029_000001""",8123596029,"""Virus genomes""",null,"""Linear""",false,"""High-quality""",0.0,"""AAI-based (high-confidence)""",100.0,0.9814,5,0,5,4559,"""vOTU_00130774""",3,"""High""","""r__Riboviria;k__Orthornavirae;p__Lenarviricota;c__Leviviricetes;o__Timlovirales;f__Steitzviridae;g__…","""Imported from NCBI""","""ssRNA(+)""","""Isolate""","""11""",null,null,"""IMGVR_UViG_8123596029_000001|8123596029|8123596029"""
"""IMGVR_UViG_8123596128_000001""",8123596128,"""Virus genomes""",null,"""Linear""",false,"""High-quality""",0.0,"""AAI-based (high-confidence)""",100.0,0.9816,5,0,5,4565,"""vOTU_00088976""",3,"""High""","""r__Riboviria;k__Orthornavirae;p__Lenarviricota;c__Leviviricetes;o__Timlovirales;f__Steitzviridae;g__…","""Imported from NCBI""","""ssRNA(+)""","""Isolate""","""11""",null,null,"""IMGVR_UViG_8123596128_000001|812

### MetaVR human gut

In [11]:
metavr_uvigs_gut_filtered = metavr_uvigs_gut.filter(metavr_quality_filter)

In [12]:
metavr_uvigs_gut_filtered

uvig,taxon_oid,discovery,discovery_doi,uvig_topology,is_concatemer,quality,checkv_contamination,completeness_method,estimated_completeness,genomad_score,uvig_cds_gene_count,uvig_trna_gene_count,uvig_total_gene_count,uvig_length,votu,n_hallmarks,viral_confidence,ictv_taxonomy,ictv_taxonomy_method,genome_type,source,genetic_code,host_taxonomy,host_taxonomy_method,fasta_header
str,i64,str,str,str,bool,str,f64,str,f64,f64,i64,i64,i64,i64,str,i64,str,str,str,str,str,str,str,str,str
"""IMGVR_UViG_2042536001_000001""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",9.88,0.9828,11,0,11,5931,"""vOTU_00679922""",3,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacillota;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae""","""iPHoP""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274"""
"""IMGVR_UViG_2042536001_000002""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",5.25,0.9834,6,0,6,5113,"""vOTU_00263288""",1,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__Crassvirales;f__Intestivirida…","""vOTU""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Chitinophagales;f__Chitinophagaceae;g__Chitinophaga;s_…","""vOTU""","""IMGVR_UViG_2042536001_000002|2042536001|Irish_EM03_contig03465"""
"""IMGVR_UViG_2042536001_000004""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",9.66,0.9826,13,0,13,5815,"""vOTU_00679916""",3,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides""","""iPHoP""","""IMGVR_UViG_2042536001_000004|2042536001|Irish_EM03_contig24613"""
"""IMGVR_UViG_2042536001_000008""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""HMM-based (lower-bound)""",11.91,0.9788,6,0,6,5544,"""vOTU_00679913""",4,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides""","""iPHoP""","""IMGVR_UViG_2042536001_000008|2042536001|Irish_EM03_contig06696"""
"""IMGVR_UViG_2042536001_000010""",2042536001,"""geNomad v1.1.0 (2022)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (medium-confidence)""",33.66,0.9695,16,0,16,21602,"""vOTU_00241963""",1,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Metagenome""","""11""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides""","""iPHoP""","""IMGVR_UViG_2042536001_000010|2042536001|Irish_EM03_contig01092"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_8122014595_000005""",8122014595,"""geNomad v1.11.0 (2025)""","""10.1038/s41587-023-01953-y""","""Linear""",false,"""Low-quality""",0.0,"""AAI-based (high-confidence)""",21.4,0.9387,16,0,16,9981,"""vOTU_08876088""",1,"""High""","""r__Duplodnaviria;k__Heunggongvirae;p__Uroviricota;c__Caudoviricetes;o__;f__;g__;s__""","""geNomad""","""dsDNA""","""Isolate""","""11""","""d__Bacteria;p__Bacillota;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Porcipelethomonas;s_…","""Isolate taxonomy""","""IMGVR_UViG_8122014595_000005|8122014595|8122014648"""
"""IMGVR_UViG_8122014595_000007""",8122014595,"""geNomad v1.11.0 (2025)""","""10.1038/s4158

## Write the output filtered sequences

In [13]:
OUT_SOIL_HEADERS = MAIN_DIR / "metavr_uvigs_soil_filtered_headers.txt"
with open(OUT_SOIL_HEADERS, "w") as out:
    for header in metavr_uvigs_soil_filtered.get_column("fasta_header").to_list():
        out.write(header + "\n")

In [14]:
! head {OUT_SOIL_HEADERS}
! wc -l {OUT_SOIL_HEADERS}

IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932
IMGVR_UViG_2088090008_000002|2088090008|P3_DRAFT_NODE_6148_len_12315_cov_71_695900
IMGVR_UViG_2088090014_000002|2088090014|GPIPI_16496951
IMGVR_UViG_2088090014_000008|2088090014|GPIPI_16804765
IMGVR_UViG_2088090014_000009|2088090014|GPIPI_16855418
IMGVR_UViG_2088090014_000010|2088090014|GPIPI_17296344
IMGVR_UViG_2088090014_000011|2088090014|GPIPI_17298922
IMGVR_UViG_2088090014_000012|2088090014|GPIPI_17305223
IMGVR_UViG_2088090014_000016|2088090014|GPIPI_16502255
IMGVR_UViG_2088090014_000017|2088090014|GPIPI_16582498
767946 /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/metavr_uvigs_soil_filtered_headers.txt


In [15]:
! seqkit grep -f {OUT_SOIL_HEADERS} {METAVR_UVIG_FNA} > {MAIN_DIR / "IMGVR5_UViG.soil.filtered.fna"}

[INFO] 767946 patterns loaded from file


In [16]:
! grep -c ">" {MAIN_DIR / "IMGVR5_UViG.soil.filtered.fna"}
! head {MAIN_DIR / "IMGVR5_UViG.soil.filtered.fna"}

767946
>IMGVR_UViG_2700989100_000001|2700989100|2700998900
CCCCCCCCCCCCCCCCCCAGGCCTCTGCGTTCTCAAGAAAGGAGTTTTGGGCGGCCTCTG
GCGTCCCGGACACGGATTGAACCTGTAGGCCGTCGTACCCATCATGATACTGCAGCAAAT
ACACTTTCATTATGCGGACTCCAACAGGAATACAATGTTTTCTACATACGCTTTCTGAAT
CGCATAGACTTCATCAAGCGTGTTGGCCACTTTCTTGTCAATACGGATTTCGGCAAGGCG
CGGTAAGAACAGAGATTTCAACGCGTCGTCCGTCTTATCCTGTACGCCATTGGAGAGCAC
TGCGGCGATCTTACCAATGTAGTCGCCCTGGTTCTCCCACATCCGGAGTCTCAGCTCATC
TGAGATCCCCGAGACGCCAACAACAAGCAAGCCATCGGAAGTCTTGCACAGAAGGGAACC
AAAGGTCTTCGCGTGTTTGCCTTTCTTGTCGGCCTCGTTGAAGCCCACGATTTCAAGGTC
ACACTCCACTTCCATTTTCAACTTCAGACCTTCGGAGGAAGTGCCATCTTCCCACGGCAT


In [17]:
OUT_GUT_HEADERS = MAIN_DIR / "metavr_uvigs_gut_filtered_headers.txt"
with open(OUT_GUT_HEADERS, "w") as out:
    for header in metavr_uvigs_gut_filtered.get_column("fasta_header").to_list():
        out.write(header + "\n")

In [18]:
! head {OUT_GUT_HEADERS}
! wc -l {OUT_GUT_HEADERS}

IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274
IMGVR_UViG_2042536001_000002|2042536001|Irish_EM03_contig03465
IMGVR_UViG_2042536001_000004|2042536001|Irish_EM03_contig24613
IMGVR_UViG_2042536001_000008|2042536001|Irish_EM03_contig06696
IMGVR_UViG_2042536001_000010|2042536001|Irish_EM03_contig01092
IMGVR_UViG_2042536001_000012|2042536001|Irish_EM03_contig24596
IMGVR_UViG_2042536001_000017|2042536001|Irish_EM03_contig24598
IMGVR_UViG_2042536001_000018|2042536001|Irish_EM03_contig24593
IMGVR_UViG_2042536001_000023|2042536001|Irish_EM03_contig23565|62382-92740
IMGVR_UViG_2042536001_000025|2042536001|Irish_EM03_contig00101|75099-126401
238034 /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/metavr_uvigs_gut_filtered_headers.txt


In [19]:
! seqkit grep -f {OUT_GUT_HEADERS} {METAVR_UVIG_FNA} > {MAIN_DIR / "IMGVR5_UViG.gut.filtered.fna"}

[INFO] 238034 patterns loaded from file


In [20]:
! grep -c ">" {MAIN_DIR / "IMGVR5_UViG.gut.filtered.fna"}
! head {MAIN_DIR / "IMGVR5_UViG.gut.filtered.fna"}

238034
>IMGVR_UViG_2706795886_000002|2706795886|2706858093
TAATTAACATATTCAACAGGAAAACCACCTAATTAGAATTGCCGACCACAAACTGCCAAA
TACTTCCCCTTTTTTCCATACTTCCTGTTTTACTCATAAATATTCATTAAATTAATTAAT
AATCACCATCATTTCCAGGGAGTGTCCTCTCTGCTATATATTCAATGACAGGTCCGAATG
GCTGAGTTTATGCCGCCAGACGGAGACGGGATCACTTCAGTGACTCCAGGCTGATCTTGG
GCGGGAGCCGAAGGTGAGTGAAACCACCGTAGTCTAGGGGCAATTCGGGCTAGATCAGTC
TGGCGGAACGGGCAAGAAACTTAAATTATTTTATTTTACAGAATGACAACAAGAATTGTA
CCTACAAAAGACAGCATTCGCCAAAAAAATCTAAAATGGATGAACCTCCTAGTACACAGC
CATGACATTTTCTGCGACTGCGACTCACCGCTACAACACACCCTTATTTTAATCTGCCAA
CAAGAACCAAAAATTGAATTAAAACCAATTGAGAAGGATATCATCAAAAAATGCCTTATT


## Run CheckAMG annotate

Since the input sequences were already quality filtered, both `--min-len` and `--min-orf` will be set to `1`.

### MetaVR soil

    nohup checkamg annotate \
        --db-dir ./CheckAMG_annotate_db_v1.1_20260316 \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_annotate_v1.1_MetaVRsoil \
        --input-contigs /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/IMGVR5_UViG.soil.filtered.fna \
        --min-len 1 \
        --min-orf 1 \
        --save-to-parquet \
        --threads 100 \
        --mem 1000 \
        > ./logs/checkamg_annotate/CheckAMG_annotate_metavr_soil.log &

In [14]:
! cat ./logs/checkamg_annotate/CheckAMG_annotate_metavr_soil.log


################################################################################
                                                                    ,-^-.
       _____  _                  _               __  __   _____     |\/\|
      / ____|| |                | |       /\    |  \/  | / ____|    `-V-'
     | |     | |__    ___   ___ | | __   /  \   | \  / || |  __       H
     | |     | '_ \  / _ \ / __|| |/ /  / /\ \  | |\/| || | |_ |      H
     | |____ | | | ||  __/| (__ |   <  / ____ \ | |  | || |__| |      H
      \_____||_| |_| \___| \___||_|\_\/_/    \_\|_|  |_| \_____|   .-;":-.
                                                                  ,'|  `; \
################################################################################

2026-07-16 19:45:29 | INFO | CheckAMG version 1.1
2026-07-16 19:45:29 | INFO | Starting CheckAMG annotate...
2026-07-16 19:45:29 | INFO | Command executed: checkamg annotate --db-dir ./CheckAMG_annotate_db_v1.1_20260316 --output /storage2/scratch/

### MetaVR human gut

    nohup checkamg annotate \
        --db-dir ./CheckAMG_annotate_db_v1.1_20260316 \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_annotate_v1.1_MetaVRgut \
        --input-contigs /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/IMGVR5_UViG.gut.filtered.fna \
        --min-len 1 \
        --min-orf 1 \
        --save-to-parquet \
        --threads 100 \
        --mem 1000 \
        > ./logs/checkamg_annotate/CheckAMG_annotate_metavr_gut.log &

In [24]:
! cat ./logs/checkamg_annotate/CheckAMG_annotate_metavr_gut.log


################################################################################
                                                                    ,-^-.
       _____  _                  _               __  __   _____     |\/\|
      / ____|| |                | |       /\    |  \/  | / ____|    `-V-'
     | |     | |__    ___   ___ | | __   /  \   | \  / || |  __       H
     | |     | '_ \  / _ \ / __|| |/ /  / /\ \  | |\/| || | |_ |      H
     | |____ | | | ||  __/| (__ |   <  / ____ \ | |  | || |__| |      H
      \_____||_| |_| \___| \___||_|\_\/_/    \_\|_|  |_| \_____|   .-;":-.
                                                                  ,'|  `; \
################################################################################

2026-07-19 09:12:35 | INFO | CheckAMG version 1.1
2026-07-19 09:12:35 | INFO | Starting CheckAMG annotate...
2026-07-19 09:12:35 | INFO | Command executed: checkamg annotate --db-dir ./CheckAMG_annotate_db_v1.1_20260316 --output /storage2/scratch/

## Run CheckAMG de-novo

The inputs are proteins, so they will need to be embedded for PST. This is done automatically by CheckAMG de-novo when proteins are the input, but this means GPUs are needed. So jobs will be submitted to the [UW-Madison Center for High Throughput Computing](https://chtc.cs.wisc.edu/) (CHTC) again, as done with the training jobs in the notebook `train_pst.ipynb`.

The pipeline invoked by `checkamg de-novo` will require:

1. Amino-acid FASTA files for the filtered soil and human gut virus sequences being used
    * Using the filtered proteins predicted by pyrodigal-GV from each CheckAMG annotate run located at `./wdir/filtered_input/filtered_faa_by_cds/single_contig_proteins.faa` for each, renamed to `IMGVR5_UViG.soil.filtered.faa` and `IMGVR5_UViG.gut.filtered.faa`
2. A pre-trained CheckAMG-PST model checkpoint (from the best model chosen in `train_pst.ipynb`) (`checkAMG-PST_TL-P__large_5.ckpt`)
3. Graph-formatted (`.h5`) PST embeddings of training proteins **generated at the end of the training workflow in `checkamg train` for the specific model checkpoint being used** (`checkAMG-PST_TL-P__large_5.PST-EMBED.h5`) **OR**
4. A precomputed train protein FAISS index constructed from the training protein PST embeddings (`checkAMG-PST_TL-P__large_5.PST-EMBED.index.faiss`) **AND**
5. Training protein labels stored in a H5 file under 'label' (`checkAMG-PST_TL-P__large_5.PST-EMBED.labels.h5`)

Either just (3) or (4 AND 5) are required. They will be included in the CheckAMG database automatically in the future but for now it will be provided manually.

### Split the inputs into smaller batches

CheckAMG de-novo should be able to handle each FASTA as single inputs, but since GPU acceleration will be used, it would be faster (in my case) to split the FASTAs into multiple files and run CheckAMG de-novo on each. This is because CheckAMG is not able to handle using multiple GPUs at once, so having multiple CheckAMG de-novo runs going concurrently for each input, each using a GPU, will be faster than running the entire input on one GPU in a single run. Will just need to be sure that proteins encoded on the same scaffold are not split across files.

In [3]:
from collections import defaultdict
from pyfastatools import Parser

N_SPLITS = 16
SPLITS_DIR = MAIN_DIR / "denovo_splits"
SPLITS_DIR.mkdir(exist_ok=True)

INPUT_FAAS = {
    "soil": MAIN_DIR / "IMGVR5_UViG.soil.filtered.faa",
    "gut": MAIN_DIR / "IMGVR5_UViG.gut.filtered.faa",
}

def scaffold_of(protein_name: str) -> str:
    # pyrodigal names proteins <scaffold>_<gene_number>
    return protein_name.rsplit("_", 1)[0]

def split_faa_by_scaffold(faa: Path, name: str, n_splits: int) -> list[Path]:
    sizes: dict[str, int] = defaultdict(int)
    for h in Parser(str(faa)).headers():
        sizes[scaffold_of(h.name)] += 1

    # LPT multiway partition: assign each scaffold (largest first) to the lightest split
    loads = [0] * n_splits
    bin_of: dict[str, int] = {}
    for scaffold, n in sorted(sizes.items(), key=lambda kv: kv[1], reverse=True):
        b = min(range(n_splits), key=loads.__getitem__)
        bin_of[scaffold] = b
        loads[b] += n

    out_paths = [SPLITS_DIR / f"IMGVR5_UViG.{name}.filtered.split{i + 1:02d}.faa" for i in range(n_splits)]
    handles = [p.open("w") for p in out_paths]
    try:
        for rec in Parser(str(faa)):
            h = rec.header
            line = h.name if not h.desc else f"{h.name} {h.desc}"
            handles[bin_of[scaffold_of(h.name)]].write(f">{line}\n{rec.seq}\n")
    finally:
        for fh in handles:
            fh.close()

    print(f"[{name}] {len(sizes):,} scaffolds, {sum(loads):,} proteins -> {n_splits} splits "
          f"(min {min(loads):,}, max {max(loads):,} proteins/split)")
    return out_paths

In [16]:
split_faa_paths = {name: split_faa_by_scaffold(faa, name, N_SPLITS) for name, faa in INPUT_FAAS.items()}
split_faa_paths

[soil] 767,786 scaffolds, 16,501,794 proteins -> 16 splits (min 1,031,362, max 1,031,363 proteins/split)
[gut] 238,034 scaffolds, 6,873,983 proteins -> 16 splits (min 429,623, max 429,624 proteins/split)


{'soil': [PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered.split01.faa'),
  PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered.split02.faa'),
  PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered.split03.faa'),
  PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered.split04.faa'),
  PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered.split05.faa'),
  PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered.split06.faa'),
  PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_splits/IMGVR5_UViG.soil.filtered

### Generate the CHTC inputs.csv for the splits

One CheckAMG de-novo job per split. Each row is `query_faa,train_index,train_labels,checkpoint,outdir,knn`, where the query FASTA is a single split and `outdir` is suffixed with the split number so concurrent jobs do not collide. The training index, labels, and model checkpoint are shared across all splits.

In [4]:
CHTC_FILES_DIR = Path("./files/checkamg_denovo")
CHTC_FILES_DIR.mkdir(exist_ok=True)
INPUTS_CSV = CHTC_FILES_DIR.joinpath("inputs.csv")

In [5]:
import numpy as np

CHECKAMG_DB_PATTERNS = "CheckAMG_denovo_db_v*_*"
db_paths = [p for p in Path(".").glob(CHECKAMG_DB_PATTERNS) if p.is_dir()]
if not db_paths:
    raise FileNotFoundError(f"No CheckAMG database directories matching {CHECKAMG_DB_PATTERNS}")
MOST_RECENT_DB = db_paths[np.argmax([p.name.split("_")[-1] for p in db_paths])]
print(f"Most recent CheckAMG_db: {MOST_RECENT_DB}")

Most recent CheckAMG_db: CheckAMG_denovo_db_v1.1_20260714


In [6]:
import glob

CKPT = Path(glob.glob(str(MOST_RECENT_DB / "*.ckpt"))[0]).name
TRAIN_INDEX = Path(glob.glob(str(MOST_RECENT_DB / "*.PST-EMBED.index.faiss"))[0]).name
TRAIN_LABELS = Path(glob.glob(str(MOST_RECENT_DB / "*.PST-EMBED.labels.h5"))[0]).name

In [7]:
knn = 20

In [22]:
with open(INPUTS_CSV, "w") as fp:
    for name in INPUT_FAAS:
        for split_faa in sorted(SPLITS_DIR.glob(f"IMGVR5_UViG.{name}.filtered.split*.faa")):
            idx = split_faa.stem.rsplit("split", 1)[-1]
            outdir = f"CheckAMG_denovo_v1.1_MetaVR{name}.split{idx}"
            row = [split_faa.name, TRAIN_INDEX, TRAIN_LABELS, CKPT, outdir, knn]
            fp.write(",".join(map(str, row)) + "\n")

In [23]:
! cat {INPUTS_CSV}

IMGVR5_UViG.soil.filtered.split01.faa,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5,checkAMG-PST_TL-P__large_5.20260714.ckpt,CheckAMG_denovo_v1.1_MetaVRsoil.split01,20
IMGVR5_UViG.soil.filtered.split02.faa,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5,checkAMG-PST_TL-P__large_5.20260714.ckpt,CheckAMG_denovo_v1.1_MetaVRsoil.split02,20
IMGVR5_UViG.soil.filtered.split03.faa,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5,checkAMG-PST_TL-P__large_5.20260714.ckpt,CheckAMG_denovo_v1.1_MetaVRsoil.split03,20
IMGVR5_UViG.soil.filtered.split04.faa,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss,checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5,checkAMG-PST_TL-P__large_5.20260714.ckpt,CheckAMG_denovo_v1.1_MetaVRsoil.split04,20
IMGVR5_UViG.soil.filtered.split05.faa,checkAMG-PST_TL-P_

## Execute jobs

After transfering the `checkamg_denovo` job files, the query protein FASTAs, the training data index/labels, and the CheckAMG-PST model checkpoint, the CheckAMG de-novo pipeline can be run. This requires a conda environment for CheckAMG (packed into a tarball with `conda-pack`) that was set up for GPU compatibility (PST installed with CUDA libraries and faiss-gpu, see [github.com/AnantharamanLab/CheckAMG](https://github.com/AnantharamanLab/CheckAMG) and [github.com/AnantharamanLab/protein_set_transformer](https://github.com/AnantharamanLab/protein_set_transformer#manually-setup-pytorch) for instructions). The training logs were deposited to `./logs/checkamg_denovo/`.

## Combine results from each CheckAMG de-novo run

Since `checkamg aggregate` expects a pair of directories, one containing `checkamg annotate` and the other containing `checkamg denovo` results, we make a single folder per dataset (soil, human gut) that holds the merged results from all of that dataset's splits.

Each combined folder contains:

* `combined_proteins.filtered.faa` (all split protein FASTAs concatenated)
* `combined_proteins.filtered.graphfmt.h5` (combined ESM2 graph-format embeddings)
* `combined_proteins.filtered.PST-EMBED.h5` (combined PST embeddings)
* `predictions.tsv` (all split predictions, single header)
* `contig_to_genome.tsv` (all split contig-to-genome maps, single header)

Every file type is combined in numeric split order (`split01`, `split02`, ...), so proteins occupy the same row order across the FASTA, both `.h5` files, and the predictions table. The `.h5` files are PyTables arrays. For `graphfmt.h5` the per-protein `data` and `strand` arrays are concatenated, `sizes` is concatenated, `genome_label` is re-indexed across splits, and the CSR `ptr` is offset by the running protein count so it stays monotonic. For `PST-EMBED.h5` the `ctx_ptn` embedding matrix is concatenated.

In [8]:
DENOVO_RUNS_DIR = MAIN_DIR / "denovo_runs"
DENOVO_RUNS_DIR.mkdir(exist_ok=True)
DENOVO_TAR_PATTERN = "CheckAMG_denovo_v1.1_MetaVR*.split*.tar.gz"

In [9]:
import subprocess

files = list(DENOVO_RUNS_DIR.glob(DENOVO_TAR_PATTERN))
processes = []

for f in files:
    print(f"Extracting {f}")
    p = subprocess.Popen([
        "tar",
        "--use-compress-program=pigz -d -p 16",
        "-xf", str(f),
        "-C", str(DENOVO_RUNS_DIR)
    ])
    processes.append((f, p))

for f, p in processes:
    ret = p.wait()
    if ret != 0:
        print(f"FAILED: {f}")
    else:
        print(f"DONE: {f}")

Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split07.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split07.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split12.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split16.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split11.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split02.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRg

In [10]:
import re
import shutil
import tables

DENOVO_DIR_SOIL_COMBINED = MAIN_DIR / "CheckAMG_denovo_v1.1_MetaVRsoil"
DENOVO_DIR_GUT_COMBINED = MAIN_DIR / "CheckAMG_denovo_v1.1_MetaVRgut"

FAA_NAME = "combined_proteins.filtered.faa"
GRAPHFMT_NAME = "combined_proteins.filtered.graphfmt.h5"
PST_EMBED_NAME = "combined_proteins.filtered.PST-EMBED.h5"
PREDICTIONS_NAME = "predictions.tsv"
CONTIG_TO_GENOME_NAME = "contig_to_genome.tsv"

def ordered_split_dirs(dataset: str) -> list[Path]:
    pattern = f"CheckAMG_denovo_v1.1_MetaVR{dataset}.split*"
    dirs = [d for d in DENOVO_RUNS_DIR.glob(pattern) if d.is_dir()]
    return sorted(dirs, key=lambda d: int(re.search(r"split(\d+)$", d.name).group(1)))

In [11]:
import numpy as np

H5_CHUNK = 50_000

def combine_faa(split_dirs: list[Path], out_path: Path) -> None:
    with open(out_path, "wb") as out:
        for d in split_dirs:
            with open(d / FAA_NAME, "rb") as fh:
                shutil.copyfileobj(fh, out)

def combine_tsv(split_dirs: list[Path], name: str, out_path: Path) -> None:
    # infer_schema_length=0 reads every column as string, preserving exact formatting
    pl.concat(
        [pl.read_csv(d / name, separator="\t", infer_schema_length=0) for d in split_dirs]
    ).write_csv(out_path, separator="\t")

def _append_in_chunks(dst, src) -> None:
    for start in range(0, src.shape[0], H5_CHUNK):
        dst.append(src[start:start + H5_CHUNK])

def combine_graphfmt(split_dirs: list[Path], out_path: Path) -> None:
    srcs = [tables.open_file(str(d / GRAPHFMT_NAME), "r") for d in split_dirs]
    try:
        ref = srcs[0].root
        with tables.open_file(str(out_path), "w") as out:
            data = out.create_earray(
                out.root, "data", atom=tables.Atom.from_dtype(ref.data.dtype),
                shape=(0, ref.data.shape[1]), filters=ref.data.filters,
            )
            strand = out.create_earray(
                out.root, "strand", atom=tables.Atom.from_dtype(ref.strand.dtype),
                shape=(0,), filters=ref.strand.filters,
            )
            comb_ptr = [0]
            comb_sizes = []
            offset = 0
            for f in srcs:
                _append_in_chunks(data, f.root.data)
                _append_in_chunks(strand, f.root.strand)
                ptr = f.root.ptr[:]
                comb_ptr.extend((ptr[1:] + offset).tolist())
                comb_sizes.append(f.root.sizes[:])
                offset += int(ptr[-1])
            comb_sizes = np.concatenate(comb_sizes)
            n_genomes = comb_sizes.shape[0]
            out.create_carray(out.root, "ptr", obj=np.asarray(comb_ptr, dtype=ref.ptr.dtype), filters=ref.ptr.filters)
            out.create_carray(out.root, "sizes", obj=comb_sizes.astype(ref.sizes.dtype), filters=ref.sizes.filters)
            out.create_carray(out.root, "genome_label", obj=np.arange(n_genomes, dtype=ref.genome_label.dtype), filters=ref.genome_label.filters)
    finally:
        for f in srcs:
            f.close()

def combine_pst_embed(split_dirs: list[Path], out_path: Path) -> None:
    with tables.open_file(str(split_dirs[0] / PST_EMBED_NAME), "r") as f0:
        atom = tables.Atom.from_dtype(f0.root.ctx_ptn.dtype)
        width = f0.root.ctx_ptn.shape[1]
        filters = f0.root.ctx_ptn.filters
    with tables.open_file(str(out_path), "w") as out:
        ctx = out.create_earray(out.root, "ctx_ptn", atom=atom, shape=(0, width), filters=filters)
        for d in split_dirs:
            with tables.open_file(str(d / PST_EMBED_NAME), "r") as f:
                _append_in_chunks(ctx, f.root.ctx_ptn)

def combine_denovo_splits(dataset: str, combined_dir: Path) -> list[Path]:
    split_dirs = ordered_split_dirs(dataset)
    if not split_dirs:
        raise FileNotFoundError(f"No extracted split directories found for dataset {dataset!r}")
    combined_dir.mkdir(exist_ok=True)
    print(f"[{dataset}] combining {len(split_dirs)} splits in order: {[d.name for d in split_dirs]}")
    combine_faa(split_dirs, combined_dir / FAA_NAME)
    combine_graphfmt(split_dirs, combined_dir / GRAPHFMT_NAME)
    combine_pst_embed(split_dirs, combined_dir / PST_EMBED_NAME)
    combine_tsv(split_dirs, PREDICTIONS_NAME, combined_dir / PREDICTIONS_NAME)
    combine_tsv(split_dirs, CONTIG_TO_GENOME_NAME, combined_dir / CONTIG_TO_GENOME_NAME)
    print(f"[{dataset}] wrote combined outputs to {combined_dir}")
    return split_dirs

In [12]:
combine_denovo_splits("soil", DENOVO_DIR_SOIL_COMBINED)

[soil] combining 16 splits in order: ['CheckAMG_denovo_v1.1_MetaVRsoil.split01', 'CheckAMG_denovo_v1.1_MetaVRsoil.split02', 'CheckAMG_denovo_v1.1_MetaVRsoil.split03', 'CheckAMG_denovo_v1.1_MetaVRsoil.split04', 'CheckAMG_denovo_v1.1_MetaVRsoil.split05', 'CheckAMG_denovo_v1.1_MetaVRsoil.split06', 'CheckAMG_denovo_v1.1_MetaVRsoil.split07', 'CheckAMG_denovo_v1.1_MetaVRsoil.split08', 'CheckAMG_denovo_v1.1_MetaVRsoil.split09', 'CheckAMG_denovo_v1.1_MetaVRsoil.split10', 'CheckAMG_denovo_v1.1_MetaVRsoil.split11', 'CheckAMG_denovo_v1.1_MetaVRsoil.split12', 'CheckAMG_denovo_v1.1_MetaVRsoil.split13', 'CheckAMG_denovo_v1.1_MetaVRsoil.split14', 'CheckAMG_denovo_v1.1_MetaVRsoil.split15', 'CheckAMG_denovo_v1.1_MetaVRsoil.split16']
[soil] wrote combined outputs to /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRsoil


[PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split01'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split02'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split03'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split04'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split05'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split06'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRsoil.split07'),

In [13]:
! mkdir -p {DENOVO_DIR_SOIL_COMBINED / "snakemake"}
! rm {DENOVO_DIR_SOIL_COMBINED / "snakemake" / "run_inference.done"}
! touch {DENOVO_DIR_SOIL_COMBINED / "snakemake" / "filter_prots.done"}
! touch {DENOVO_DIR_SOIL_COMBINED / "snakemake" / "compute_query_esm2.done"}
! touch {DENOVO_DIR_SOIL_COMBINED / "snakemake" / "compute_query_pst.done"}

rm: cannot remove '/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRsoil/snakemake/run_inference.done': No such file or directory


    checkamg de-novo \
        --query-proteins /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/IMGVR5_UViG.soil.filtered.faa \
        --train-index-file /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss \
        --train-labels-file /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5 \
        --model-ckpt /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.ckpt \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRsoil/ \
        --esm2-ckpt-dir /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/checkpoints/ \
        --knn 20 \
        -a cpu \
        --devices 200 \
        --mem 1000

In [14]:
combine_denovo_splits("gut", DENOVO_DIR_GUT_COMBINED)

[gut] combining 16 splits in order: ['CheckAMG_denovo_v1.1_MetaVRgut.split01', 'CheckAMG_denovo_v1.1_MetaVRgut.split02', 'CheckAMG_denovo_v1.1_MetaVRgut.split03', 'CheckAMG_denovo_v1.1_MetaVRgut.split04', 'CheckAMG_denovo_v1.1_MetaVRgut.split05', 'CheckAMG_denovo_v1.1_MetaVRgut.split06', 'CheckAMG_denovo_v1.1_MetaVRgut.split07', 'CheckAMG_denovo_v1.1_MetaVRgut.split08', 'CheckAMG_denovo_v1.1_MetaVRgut.split09', 'CheckAMG_denovo_v1.1_MetaVRgut.split10', 'CheckAMG_denovo_v1.1_MetaVRgut.split11', 'CheckAMG_denovo_v1.1_MetaVRgut.split12', 'CheckAMG_denovo_v1.1_MetaVRgut.split13', 'CheckAMG_denovo_v1.1_MetaVRgut.split14', 'CheckAMG_denovo_v1.1_MetaVRgut.split15', 'CheckAMG_denovo_v1.1_MetaVRgut.split16']


[gut] wrote combined outputs to /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRgut


[PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split01'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split02'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split03'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split04'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split05'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split06'),
 PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/denovo_runs/CheckAMG_denovo_v1.1_MetaVRgut.split07'),
 Posix

In [15]:
! mkdir -p {DENOVO_DIR_GUT_COMBINED / "snakemake"}
! rm {DENOVO_DIR_GUT_COMBINED / "snakemake" / "run_inference.done"}
! touch {DENOVO_DIR_GUT_COMBINED / "snakemake" / "filter_prots.done"}
! touch {DENOVO_DIR_GUT_COMBINED / "snakemake" / "compute_query_esm2.done"}
! touch {DENOVO_DIR_GUT_COMBINED / "snakemake" / "compute_query_pst.done"}

rm: cannot remove '/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRgut/snakemake/run_inference.done': No such file or directory


    checkamg de-novo \
        --query-proteins /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/IMGVR5_UViG.gut.filtered.faa \
        --train-index-file /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss \
        --train-labels-file /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5 \
        --model-ckpt /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.ckpt \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRgut \
        --esm2-ckpt-dir /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/checkpoints/ \
        --knn 20 \
        -a cpu \
        --devices 200 \
        --mem 1000

## Run CheckAMG aggregate

### MetaVR soil

    nohup checkamg aggregate \
        --annotate-dir /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_annotate_v1.1_MetaVRsoil \
        --denovo-dir /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRsoil \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_aggregate_v1.1_MetaVRsoil \
        --save-to-parquet \
        --threads 100 \
        > ./logs/checkamg_aggregate/CheckAMG_aggregate_metavr_soil.log &

### MetaVR human gut

    nohup checkamg aggregate \
        --annotate-dir /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_annotate_v1.1_MetaVRgut \
        --denovo-dir /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_denovo_v1.1_MetaVRgut \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/CheckAMG_aggregate_v1.1_MetaVRgut \
        --save-to-parquet \
        --threads 100 \
        > ./logs/checkamg_aggregate/CheckAMG_aggregate_metavr_gut.log &

## Load the aggregated results

Separatley load and assign confidence bin levels, in case the cutoffs changed.

In [16]:
import json
pst_thresholds = json.load(open("../CheckAMG/files/pst_thresholds.json"))
pst_thresholds

{'AVG-like': {'very_high': 0.89, 'high': 0.201, 'medium': 0.034},
 'Viral': {'very_high': 0.622, 'high': 0.051, 'medium': 0.001},
 'AVG': {'very_high': 0.956, 'high': 0.512, 'medium': 0.096}}

In [17]:
AGGREGATE_DIR_SOIL = MAIN_DIR / "CheckAMG_aggregate_v1.1_MetaVRsoil"
AGGREGATE_DIR_GUT = MAIN_DIR / "CheckAMG_aggregate_v1.1_MetaVRgut"

In [18]:
def pst_confidence_expr(prob_col, cutoffs):
    # explicit null guard so missing de-novo probabilities stay null instead of falling through to "low"
    return (
        pl.when(pl.col(prob_col).is_null()).then(pl.lit(None))
        .when(pl.col(prob_col) >= cutoffs["very_high"]).then(pl.lit("very high"))
        .when(pl.col(prob_col) >= cutoffs["high"]).then(pl.lit("high"))
        .when(pl.col(prob_col) >= cutoffs["medium"]).then(pl.lit("medium"))
        .otherwise(pl.lit("low"))
    )

agg_res_detailed_soil = (
    pl.read_parquet(AGGREGATE_DIR_SOIL / "aggregated_results_detailed.parquet")
    .with_columns([
        pst_confidence_expr("Viral Probability (de-novo)", pst_thresholds["Viral"]).alias("Viral Confidence Level (de-novo)"),
        pst_confidence_expr("AVG-like Probability (de-novo)", pst_thresholds["AVG-like"]).alias("AVG-like Confidence Level (de-novo)"),
        pst_confidence_expr("Final AVG Probability (de-novo)", pst_thresholds["AVG"]).alias("Final AVG Confidence Level (de-novo)"),
    ])
)
agg_res_detailed_soil

Protein,Contig,Genome,Viral Probability (annotate),Viral Confidence Level (annotate),Viral Probability (de-novo),Viral Confidence Level (de-novo),Classification (annotate),Function (annotate),AVG-like Probability (de-novo),AVG-like Confidence Level (de-novo),Final AVG Probability (de-novo),Final AVG Confidence Level (de-novo)
str,str,str,f32,str,f64,str,str,str,f64,str,f64,str
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_1""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.965619,"""high""",1.0,"""very high""","""unclassified""","""""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_2""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.954897,"""high""",1.0,"""very high""","""unclassified""","""terminase small subunit""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_3""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.969051,"""high""",1.0,"""very high""","""unclassified""","""terminase large subunit""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_4""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.959374,"""high""",1.0,"""very high""","""unclassified""","""head-tail joining""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_5""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.985477,"""high""",1.0,"""very high""","""unclassified""","""Phage portal protein, lambda family""",0.0,"""low""",0.0,"""low"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_8123603135_000001|8123603135|8123603135_88""","""IMGVR_UViG_8123603135_000001|8123603135|8123603135""","""IMGVR_UViG_8123603135_000001|8123603135|8123603135""",0.986737,"""high""",1.0,"""very high""","""unclassified""",null,0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_8123603135_000001|8123603135|8123603135_89""","""IMGVR_UViG_8123603135_000001|8123603135|8123603135""","""IMGVR_UViG_8123603135_000001|8123603135|8123603135""",0.987233,"""high""",1.0,"""very high""","""unclassified""","""""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_8123603135_000001|8123603135|8123603135_90""","""IMGVR_UViG_8123603135_000001|8123603135|8123603135""","""IMGVR_UViG_8123603135_000001|8123603135|8123603135""",0.988478,"""high""",1.0,"""very high""","""unclassified""","""""",0.0,"""low""",0.0,"""low"""


In [19]:
agg_res_detailed_gut = (
    pl.read_parquet(AGGREGATE_DIR_GUT / "aggregated_results_detailed.parquet")
    .with_columns([
        pst_confidence_expr("Viral Probability (de-novo)", pst_thresholds["Viral"]).alias("Viral Confidence Level (de-novo)"),
        pst_confidence_expr("AVG-like Probability (de-novo)", pst_thresholds["AVG-like"]).alias("AVG-like Confidence Level (de-novo)"),
        pst_confidence_expr("Final AVG Probability (de-novo)", pst_thresholds["AVG"]).alias("Final AVG Confidence Level (de-novo)"),
    ])
)
agg_res_detailed_gut

Protein,Contig,Genome,Viral Probability (annotate),Viral Confidence Level (annotate),Viral Probability (de-novo),Viral Confidence Level (de-novo),Classification (annotate),Function (annotate),AVG-like Probability (de-novo),AVG-like Confidence Level (de-novo),Final AVG Probability (de-novo),Final AVG Confidence Level (de-novo)
str,str,str,f32,str,f64,str,str,str,f64,str,f64,str
"""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274_1""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""",0.961994,"""high""",1.0,"""very high""","""unclassified""",null,0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274_2""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""",0.956285,"""high""",1.0,"""very high""","""unclassified""","""Domain of unknown function (DUF7253)""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274_3""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""",0.947419,"""high""",1.0,"""very high""","""unclassified""",null,0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274_4""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""",0.949114,"""high""",1.0,"""very high""","""unclassified""","""major tail protein""",0.0,"""low""",0.0,"""low"""
"""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274_5""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""","""IMGVR_UViG_2042536001_000001|2042536001|Irish_EM03_contig02274""",0.938472,"""high""",1.0,"""very high""","""unclassified""","""""",0.0,"""low""",0.0,"""low"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_8123603117_000001|8123603117|8123603117_1""","""IMGVR_UViG_8123603117_000001|8123603117|8123603117""","""IMGVR_UViG_8123603117_000001|8123603117|8123603117""",0.99515,"""high""",null,null,"""unclassified""",null,null,null,null,null
"""IMGVR_UViG_8123603334_000001|8123603334|8123603334_1""","""IMGVR_UViG_8123603334_000001|8123603334|8123603334""","""IMGVR_UViG_8123603334_000001|8123603334|8123603334""",0.99515,"""high""",null,null,"""unclassified""",null,null,null,null,null
"""IMGVR_UViG_3300000287_000055|3300000287|EM272_1050227_16""","""IMGVR_UViG_3300000287_000055|3300000287|EM272_1050227""","""IMGVR_UViG_3300000287_000055|3300000287|EM272_1050227""",null,null,1.0,"""very high""",null,null,0.0,"""low""",0.0,"""low"""


In [20]:
agg_res_detailed_combined = pl.concat([
    agg_res_detailed_soil.with_columns(pl.lit("soil").alias("dataset")),
    agg_res_detailed_gut.with_columns(pl.lit("gut").alias("dataset"))
    ])
agg_res_detailed_combined

Protein,Contig,Genome,Viral Probability (annotate),Viral Confidence Level (annotate),Viral Probability (de-novo),Viral Confidence Level (de-novo),Classification (annotate),Function (annotate),AVG-like Probability (de-novo),AVG-like Confidence Level (de-novo),Final AVG Probability (de-novo),Final AVG Confidence Level (de-novo),dataset
str,str,str,f32,str,f64,str,str,str,f64,str,f64,str,str
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_1""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.965619,"""high""",1.0,"""very high""","""unclassified""","""""",0.0,"""low""",0.0,"""low""","""soil"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_2""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.954897,"""high""",1.0,"""very high""","""unclassified""","""terminase small subunit""",0.0,"""low""",0.0,"""low""","""soil"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_3""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.969051,"""high""",1.0,"""very high""","""unclassified""","""terminase large subunit""",0.0,"""low""",0.0,"""low""","""soil"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_4""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.959374,"""high""",1.0,"""very high""","""unclassified""","""head-tail joining""",0.0,"""low""",0.0,"""low""","""soil"""
"""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932_5""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""","""IMGVR_UViG_2088090008_000001|2088090008|P3_DRAFT_NODE_290175_len_15711_cov_24_196932""",0.985477,"""high""",1.0,"""very high""","""unclassified""","""Phage portal protein, lambda family""",0.0,"""low""",0.0,"""low""","""soil"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_8123603117_000001|8123603117|8123603117_1""","""IMGVR_UViG_8123603117_000001|8123603117|8123603117""","""IMGVR_UViG_8123603117_000001|8123603117|8123603117""",0.99515,"""high""",null,null,"""unclassified""",null,null,null,null,null,"""gut"""
"""IMGVR_UViG_8123603334_000001|8123603334|8123603334_1""","""IMGVR_UViG_8123603334_000001|8123603334|8123603334""","""IMGVR_UViG_8123603334_000001|8123603334|8123603334""",0.99515,"""high""",null,null,"""unclassified""",null,null,null,null,null,"""gut"""
"""IMGVR_UViG_3300000287_000055|3300000287|EM272_1050227_16""","""IMGVR_UViG_3300000287_000055|3300000287|EM272_1050227""","""IMGVR_UViG_3300000287_000055|3300000287|EM272_1050227""",null,null,1.0,"""very high""",null,null,0.0,"""low""",0.0,"""low""","""gut"""


In [21]:
agg_res_detailed_combined.write_parquet(MAIN_DIR / "aggregated_results_combined.parquet")

In [22]:
agg_res_detailed_combined.filter((pl.col("Classification (annotate)") == "unclassified") & (pl.col("Final AVG Confidence Level (de-novo)").is_in(["high", "very high"])))

Protein,Contig,Genome,Viral Probability (annotate),Viral Confidence Level (annotate),Viral Probability (de-novo),Viral Confidence Level (de-novo),Classification (annotate),Function (annotate),AVG-like Probability (de-novo),AVG-like Confidence Level (de-novo),Final AVG Probability (de-novo),Final AVG Confidence Level (de-novo),dataset
str,str,str,f32,str,f64,str,str,str,f64,str,f64,str,str
"""IMGVR_UViG_2088090015_000001|2088090015|GPICI_5395867_16""","""IMGVR_UViG_2088090015_000001|2088090015|GPICI_5395867""","""IMGVR_UViG_2088090015_000001|2088090015|GPICI_5395867""",0.99721,"""high""",1.0,"""very high""","""unclassified""",null,0.955,"""very high""",0.955,"""high""","""soil"""
"""IMGVR_UViG_2088090015_000074|2088090015|GPICI_8944106_6""","""IMGVR_UViG_2088090015_000074|2088090015|GPICI_8944106""","""IMGVR_UViG_2088090015_000074|2088090015|GPICI_8944106""",0.998103,"""high""",1.0,"""very high""","""unclassified""",null,0.6442,"""high""",0.6442,"""high""","""soil"""
"""IMGVR_UViG_2088090015_000101|2088090015|GPICI_9056187_71""","""IMGVR_UViG_2088090015_000101|2088090015|GPICI_9056187""","""IMGVR_UViG_2088090015_000101|2088090015|GPICI_9056187""",0.993693,"""high""",1.0,"""very high""","""unclassified""","""""",0.7429,"""high""",0.7429,"""high""","""soil"""
"""IMGVR_UViG_2088090015_000294|2088090015|GPICI_8678465_10""","""IMGVR_UViG_2088090015_000294|2088090015|GPICI_8678465""","""IMGVR_UViG_2088090015_000294|2088090015|GPICI_8678465""",0.521914,"""medium""",1.0,"""very high""","""unclassified""","""Enoyl-(Acyl carrier protein) reductase""",0.8974,"""very high""",0.8974,"""high""","""soil"""
"""IMGVR_UViG_2088090015_000371|2088090015|GPICI_8802683_6""","""IMGVR_UViG_2088090015_000371|2088090015|GPICI_8802683""","""IMGVR_UViG_2088090015_000371|2088090015|GPICI_8802683""",0.845711,"""medium""",1.0,"""very high""","""unclassified""","""AAA domain""",0.6038,"""high""",0.6038,"""high""","""soil"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_8120862049_000001|8120862049|8120862136|6313-42244_26""","""IMGVR_UViG_8120862049_000001|8120862049|8120862136|6313-42244""","""IMGVR_UViG_8120862049_000001|8120862049|8120862136|6313-42244""",0.998714,"""high""",1.0,"""very high""","""unclassified""",null,0.648,"""high""",0.648,"""high""","""gut"""
"""IMGVR_UViG_8121143641_000002|8121143641|8121143641|3750603-3792852_22""","""IMGVR_UViG_8121143641_000002|8121143641|8121143641|3750603-3792852""","""IMGVR_UViG_8121143641_000002|8121143641|8121143641|3750603-3792852""",0.995395,"""high""",1.0,"""very high""","""unclassified""","""""",1.0,"""very high""",1.0,"""very high""","""gut"""
"""IMGVR_UViG_8121206275_000002|8121206275|8121206275|2595527-2631438_29""","""IMGVR_UViG_8121206275_000002|8121206275|8121206275|2595527-2631438""","""IMGVR_UViG_8121206275_000002|8121206275|8121206275|2595527-2631438""",0.99502,"""high""",1.0,"""very high""","""unclassified""","""toxin""",1.0,"""very high""",1.0,"""very high""","""gut"""
